# 06 - LLM-Based Retention Message Generation

Selects 50 at-risk profiles from the Hybrid XGBoost predictions, builds a SHAP-grounded prompt, generates two messages (LLM and generic), and aggregates the human-evaluation Likert scores.

In [ ]:
import sys, pathlib, joblib, numpy as np, pandas as pd
sys.path.insert(0, str(pathlib.Path('.').resolve().parent))
from code import config, retention_llm, shap_explain

art = joblib.load(config.PROCESSED_DIR / 'best_hybrid_xgb.joblib')
pipe, names, X_test, y_test = art['pipe'], art['feature_names'], art['X_test'], art['y_test']

merged = pd.read_parquet(config.PROCESSED_DIR / 'bank_merged.parquet')
y_prob = pipe.predict_proba(X_test)[:, 1]
at_risk_idx = np.where(y_prob >= config.AT_RISK_THRESHOLD)[0][:50]
print('At-risk customers selected:', len(at_risk_idx))

In [ ]:
# Compute SHAP for the at-risk subset and build prompts
X_pre = pipe[:-1].transform(X_test[at_risk_idx])
sv = shap_explain.explain_tree(pipe.named_steps['clf'], X_pre, names)

POLICY = (
    'May offer fee waivers, complimentary advisory sessions, or product-tier upgrades. '
    'Must not promise specific monetary amounts. '
    'Always include a clear next step.'
)
prompts = []
for i in range(len(at_risk_idx)):
    row = merged.iloc[at_risk_idx[i]]
    profile = retention_llm.profile_blurb(row)
    contribs = sorted(zip(names, sv.values[i], sv.data[i]), key=lambda t: -abs(t[1]))[:5]
    drivers = [retention_llm.driver_to_phrase(n, v) for n, _, v in contribs]
    prompts.append(retention_llm.build_prompt(profile, drivers, POLICY))
print(prompts[0])

In [ ]:
# Generation - configure your backend (OpenAI or local Llama 3 / Mistral 4-bit).
BACKEND = 'openai'  # or 'hf'
messages = []
for p in prompts:
    if BACKEND == 'openai':
        m = retention_llm.generate_with_openai(p)
    else:
        m = retention_llm.generate_with_hf(p)
    messages.append({'kind': 'llm', 'message': m})
    messages.append({'kind': 'generic', 'message': retention_llm.build_generic_message()})
pd.DataFrame(messages).to_csv(config.PROCESSED_DIR / 'retention_messages.csv', index=False)
print('Generated', len(messages), 'messages')

In [ ]:
# After human evaluation, load scores and summarise
scores_path = config.PROCESSED_DIR / 'retention_human_scores.csv'
if scores_path.exists():
    scores = pd.read_csv(scores_path)
    summary = retention_llm.summarise_human_scores(scores)
    summary.to_csv(config.PROCESSED_DIR / 'table_4_9_human_eval.csv', index=False)
    print(summary)
else:
    print('Human evaluation scores not found; collect via the rubric in Appendix B and save as retention_human_scores.csv (columns: profile_id, evaluator_id, kind, Relevance, Personalization, Actionability, Persuasiveness).')